<a href="https://colab.research.google.com/github/tangitapkullaniyor/CENG467_Midterm_290201060/blob/main/Question3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets transformers evaluate rouge_score nltk bert-score networkx scikit-learn -q

In [ ]:
from datasets import load_dataset

dataset_sum = load_dataset("cnn_dailymail", "3.0.0")

print(dataset_sum)
print(dataset_sum["test"][0].keys())
print(dataset_sum["test"][0]["article"][:500])
print(dataset_sum["test"][0]["highlights"])

In [ ]:
test_data = dataset_sum["test"].select(range(50))

In [ ]:
import nltk
nltk.download("punkt")
nltk.download('punkt_tab')

In [ ]:
import numpy as np
import networkx as nx
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def textrank_summary(text, num_sentences=3):
    sentences = sent_tokenize(text)

    if len(sentences) <= num_sentences:
        return " ".join(sentences)

    vectorizer = TfidfVectorizer(stop_words="english")
    sentence_vectors = vectorizer.fit_transform(sentences)

    similarity_matrix = cosine_similarity(sentence_vectors)

    graph = nx.from_numpy_array(similarity_matrix)
    scores = nx.pagerank(graph)

    ranked_sentences = sorted(
        ((scores[i], sentence) for i, sentence in enumerate(sentences)),
        reverse=True
    )

    selected_sentences = [sentence for _, sentence in ranked_sentences[:num_sentences]]

    return " ".join(selected_sentences)

In [ ]:
!pip install sentencepiece -q

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_checkpoint = "t5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(t5_checkpoint)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_checkpoint)

In [ ]:
def t5_summary(text, max_input_length=512, max_output_length=80):
    input_text = "summarize: " + text

    inputs = t5_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True
    )

    summary_ids = t5_model.generate(
        inputs["input_ids"],
        max_length=max_output_length,
        min_length=30,
        num_beams=4,
        early_stopping=True
    )

    return t5_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [ ]:
for i in range(3):
    article = test_data[i]["article"]
    reference = test_data[i]["highlights"]

    textrank_sum = textrank_summary(article, num_sentences=3)
    t5_sum = t5_summary(article)

    print("=" * 80)
    print(f"EXAMPLE {i+1}")
    print("=" * 80)

    print("\nARTICLE SNIPPET:")
    print(article[:500])

    print("\nREFERENCE SUMMARY:")
    print(reference)

    print("\nTEXTRANK SUMMARY:")
    print(textrank_sum)

    print("\nT5 SUMMARY:")
    print(t5_sum)

    print("\n")

In [ ]:
!pip install evaluate rouge_score bert-score nltk -q

In [ ]:
import evaluate

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

In [ ]:
references = []
textrank_preds = []
t5_preds = []

for example in test_data:
    article = example["article"]
    reference = example["highlights"]

    textrank_sum = textrank_summary(article)
    t5_sum = t5_summary(article)

    references.append(reference)
    textrank_preds.append(textrank_sum)
    t5_preds.append(t5_sum)

In [ ]:
rouge_textrank = rouge.compute(predictions=textrank_preds, references=references)
rouge_t5 = rouge.compute(predictions=t5_preds, references=references)

print("TextRank ROUGE:", rouge_textrank)
print("T5 ROUGE:", rouge_t5)

In [ ]:
bleu_textrank = bleu.compute(
    predictions=textrank_preds,
    references=[[ref] for ref in references]
)

bleu_t5 = bleu.compute(
    predictions=t5_preds,
    references=[[ref] for ref in references]
)

print("TextRank BLEU:", bleu_textrank)
print("T5 BLEU:", bleu_t5)

In [ ]:
meteor_textrank = meteor.compute(
    predictions=textrank_preds,
    references=references
)

meteor_t5 = meteor.compute(
    predictions=t5_preds,
    references=references
)

print("TextRank METEOR:", meteor_textrank)
print("T5 METEOR:", meteor_t5)

In [ ]:
bertscore_textrank = bertscore.compute(
    predictions=textrank_preds,
    references=references,
    lang="en"
)

bertscore_t5 = bertscore.compute(
    predictions=t5_preds,
    references=references,
    lang="en"
)

print("TextRank BERTScore F1:", sum(bertscore_textrank["f1"]) / len(bertscore_textrank["f1"]))
print("T5 BERTScore F1:", sum(bertscore_t5["f1"]) / len(bertscore_t5["f1"]))